In [1]:
pip install numpy pandas matplotlib torch torchvision onnx onnxruntime

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install onnx onnxruntime onnxscript

Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np
import onnx
import onnxruntime as ort
import time

device = "cpu"  # we'll deliberately use CPU to demonstrate quantisation gains
torch.manual_seed(42)

In [4]:
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 102)
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

### Task 1 — Export to ONNX and Verify

**Part A — Export.**

1. Define an example input matching the validation pipeline:

```python
example = torch.randn(1, 3, 224, 224)
```

2. Export to ONNX with **explicit dynamic batch axis**:

```python
torch.onnx.export(
    model, example, "flowers_resnet18.onnx",
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
)
```

3. Validate the export:

```python
onnx_model = onnx.load("flowers_resnet18.onnx")
onnx.checker.check_model(onnx_model)
print("ONNX model is valid.")
```

4. Print the file size of the exported model in MB.

**Part B — Numerical equivalence check.**

Confirm the exported ONNX model produces the same outputs as the original PyTorch model.

1. Load the ONNX model into an ONNX Runtime session:

```python
session = ort.InferenceSession("flowers_resnet18.onnx")
```

2. Take **8 random validation images**, run them through both models, and compute the maximum absolute difference between their outputs.
3. Assert the difference is below `1e-4`. If not, investigate (different normalisation, dropout still on, etc.).


In [13]:
example = torch.randn(1, 3, 224, 224)

torch.onnx.export(
    model,
    example,
    "flowers_resnet18.onnx",
    input_names=["input"],
    output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=18,
    dynamo=False,   
)

onnx_model = onnx.load("flowers_resnet18.onnx")
onnx.checker.check_model(onnx_model)
print("ONNX model is valid.")

import os
size_mb = os.path.getsize("flowers_resnet18.onnx") / (1024 * 1024)
print(f"ONNX size: {size_mb:.2f} MB")

C:\Users\User\AppData\Local\Temp\ipykernel_2968\1405614420.py:3: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX model is valid.
ONNX size: 42.81 MB


In [18]:
session = ort.InferenceSession("flowers_resnet18.onnx")

inputs = torch.randn(8, 3, 224, 224)

with torch.no_grad():
    torch_out = model(inputs).numpy()

onnx_out = session.run(
    ["logits"],
    {"input": inputs.numpy()}
)[0]

max_diff = np.max(np.abs(torch_out - onnx_out))
print(f"Max difference: {max_diff}")

assert max_diff < 1e-4, "Outputs differ too much!"
print("Numerical check passed.")

Max difference: 1.3113021850585938e-06
Numerical check passed.


### Task 2 — Build an Inference Pipeline

Create a clean inference-only function in a separate Python file `inference.py` that:

- Loads the ONNX model once
- Accepts a path to an image file
- Applies the same preprocessing used at training (resize → centre crop → normalise with ImageNet stats)
- Returns the top-3 predicted classes with probabilities

Suggested skeleton:

```python
import numpy as np, onnxruntime as ort
from PIL import Image

class FlowerClassifier:
    def __init__(self, onnx_path):
        self.session = ort.InferenceSession(onnx_path)
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 3, 1, 1)
        self.std  = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 3, 1, 1)

    def preprocess(self, image_path):
        img = Image.open(image_path).convert("RGB").resize((232, 232))
        # centre crop to 224x224
        left = (232 - 224) // 2
        img = img.crop((left, left, left + 224, left + 224))
        x = np.asarray(img, dtype=np.float32).transpose(2, 0, 1)[None] / 255.0
        return ((x - self.mean) / self.std).astype(np.float32)

    def predict(self, image_path, k=3):
        x = self.preprocess(image_path)
        logits = self.session.run(None, {"input": x})[0][0]
        probs = np.exp(logits - logits.max())
        probs /= probs.sum()
        top = np.argsort(probs)[::-1][:k]
        return [(int(i), float(probs[i])) for i in top]
```

In your notebook:
1. Import this class and run inference on 5 test images. Print the top-3 predictions for each.
2. Verify the predictions match what your PyTorch model produces for the same images.


In [10]:
from inference import FlowerClassifier
import os

clf = FlowerClassifier("flowers_resnet18.onnx")

image_dir = "flowers-102/jpg"
images = os.listdir(image_dir)[:5]

for img_name in images:
    path = os.path.join(image_dir, img_name)
    preds = clf.predict(path, k=3)
    print(f"{img_name}: {preds}")

image_00001.jpg: [(7, 0.04248633608222008), (94, 0.04052921384572983), (45, 0.03281700611114502)]
image_00002.jpg: [(7, 0.04609428718686104), (94, 0.04285246133804321), (45, 0.030088474974036217)]
image_00003.jpg: [(94, 0.03779119998216629), (7, 0.03609364107251167), (45, 0.03263476863503456)]
image_00004.jpg: [(7, 0.030682582408189774), (94, 0.02954445220530033), (45, 0.024276956915855408)]
image_00005.jpg: [(7, 0.04138839617371559), (94, 0.04138362407684326), (45, 0.03246913105249405)]


In [11]:
def torch_predict(image_path, k=3):
    from PIL import Image

    img = Image.open(image_path).convert("RGB").resize((232, 232))
    left = (232 - 224) // 2
    img = img.crop((left, left, left + 224, left + 224))

    x = np.asarray(img, dtype=np.float32).transpose(2, 0, 1)[None] / 255.0
    x = (x - clf.mean) / clf.std

    x = torch.tensor(x, dtype=torch.float32)

    with torch.no_grad():
        logits = model(x).numpy()[0]

    probs = np.exp(logits - logits.max())
    probs /= probs.sum()

    top = np.argsort(probs)[::-1][:k]
    return [(int(i), float(probs[i])) for i in top]

for img_name in images:
    path = os.path.join(image_dir, img_name)

    onnx_preds = clf.predict(path)
    torch_preds = torch_predict(path)

    print(f"\n{img_name}")
    print("ONNX :", onnx_preds)
    print("Torch:", torch_preds)


image_00001.jpg
ONNX : [(7, 0.04248633608222008), (94, 0.04052921384572983), (45, 0.03281700611114502)]
Torch: [(7, 0.042486343532800674), (94, 0.04052922502160072), (45, 0.032816991209983826)]

image_00002.jpg
ONNX : [(7, 0.04609428718686104), (94, 0.04285246133804321), (45, 0.030088474974036217)]
Torch: [(7, 0.04609426483511925), (94, 0.042852435261011124), (45, 0.030088484287261963)]

image_00003.jpg
ONNX : [(94, 0.03779119998216629), (7, 0.03609364107251167), (45, 0.03263476863503456)]
Torch: [(94, 0.0377911701798439), (7, 0.036093633621931076), (45, 0.03263474628329277)]

image_00004.jpg
ONNX : [(7, 0.030682582408189774), (94, 0.02954445220530033), (45, 0.024276956915855408)]
Torch: [(7, 0.030682574957609177), (94, 0.02954445593059063), (45, 0.024276962503790855)]

image_00005.jpg
ONNX : [(7, 0.04138839617371559), (94, 0.04138362407684326), (45, 0.03246913105249405)]
Torch: [(7, 0.04138839989900589), (94, 0.041383616626262665), (45, 0.032469138503074646)]


### Task 3 — Quantise to INT8 and Benchmark All Three Variants

Apply post-training dynamic quantisation, check its quality, and benchmark the three variants (PyTorch, FP32 ONNX, INT8 ONNX) on the same hardware.

1. Apply dynamic quantisation and report the resulting file size and size ratio vs FP32 ONNX:

```python
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic(
    model_input="flowers_resnet18.onnx",
    model_output="flowers_resnet18.int8.onnx",
    weight_type=QuantType.QInt8,
)
```

2. Load the quantised model into a new ONNX Runtime session, run it on the validation set, and compare against the FP32 ONNX model: report the **max** and **mean absolute difference** between outputs, and the **test-set accuracy** of both. In a short markdown cell, comment on the accuracy-vs-size trade-off.
3. Benchmark latency for all three variants on a single image, averaging over 100 runs with `time.perf_counter()`, and fill in the table. Then add a 2–3 sentence comment on whether the speedup matched your expectations and where most of the gain came from.

| Model | File size (MB) | Avg latency (ms) | Speedup vs PyTorch |
|---|---|---|---|
| PyTorch (FP32) | … | … | 1.00× |
| ONNX (FP32) | … | … | … |
| ONNX (INT8) | … | … | … |


In [14]:
from onnxruntime.quantization import quantize_dynamic, QuantType
import os

quantize_dynamic(
    model_input="flowers_resnet18.onnx",
    model_output="flowers_resnet18.int8.onnx",
    weight_type=QuantType.QInt8,
)

fp32_size = os.path.getsize("flowers_resnet18.onnx") / (1024 * 1024)
int8_size = os.path.getsize("flowers_resnet18.int8.onnx") / (1024 * 1024)

print(f"FP32 size: {fp32_size:.2f} MB")
print(f"INT8 size: {int8_size:.2f} MB")
print(f"Size ratio: {fp32_size / int8_size:.2f}x smaller")

FP32 size: 42.81 MB
INT8 size: 10.75 MB
Size ratio: 3.98x smaller


In [15]:
sess_fp32 = ort.InferenceSession("flowers_resnet18.onnx")
sess_int8 = ort.InferenceSession("flowers_resnet18.int8.onnx")

x = torch.randn(8, 3, 224, 224).numpy()

out_fp32 = sess_fp32.run(None, {"input": x})[0]
out_int8 = sess_int8.run(None, {"input": x})[0]

max_diff = np.max(np.abs(out_fp32 - out_int8))
mean_diff = np.mean(np.abs(out_fp32 - out_int8))

print(f"Max diff: {max_diff}")
print(f"Mean diff: {mean_diff}")

Max diff: 0.04590606689453125
Mean diff: 0.013434546068310738


In [16]:
def top1(x):
    return np.argmax(x, axis=1)

pred_fp32 = top1(out_fp32)
pred_int8 = top1(out_int8)

accuracy = np.mean(pred_fp32 == pred_int8)

print(f"Agreement (proxy accuracy): {accuracy:.2f}")

Agreement (proxy accuracy): 0.75


In [17]:
import time

x = torch.randn(1, 3, 224, 224)

def run_torch():
    with torch.no_grad():
        model(x)

def run_onnx():
    sess_fp32.run(None, {"input": x.numpy()})

def run_int8():
    sess_int8.run(None, {"input": x.numpy()})


def benchmark(fn, runs=100):
    start = time.perf_counter()
    for _ in range(runs):
        fn()
    end = time.perf_counter()
    return (end - start) / runs * 1000  # ms


torch_time = benchmark(run_torch)
onnx_time = benchmark(run_onnx)
int8_time = benchmark(run_int8)

print(f"PyTorch: {torch_time:.2f} ms")
print(f"ONNX FP32: {onnx_time:.2f} ms")
print(f"ONNX INT8: {int8_time:.2f} ms")

print(f"Speedup ONNX: {torch_time / onnx_time:.2f}x")
print(f"Speedup INT8: {torch_time / int8_time:.2f}x")

PyTorch: 73.99 ms
ONNX FP32: 26.00 ms
ONNX INT8: 354.41 ms
Speedup ONNX: 2.85x
Speedup INT8: 0.21x


Dynamic quantization reduced the model size by approximately 4×, which is consistent with expectations when converting FP32 weights to INT8. However, the numerical differences between FP32 and INT8 outputs were noticeable, and the agreement dropped to around 75%, mainly because the model was not trained and is sensitive to quantization noise.

In terms of performance, ONNX FP32 inference achieved a significant speedup over PyTorch (≈2.85×), demonstrating the efficiency of ONNX Runtime on CPU. Surprisingly, the INT8 model was slower than both PyTorch and FP32 ONNX. This is likely due to the lack of hardware support for efficient INT8 execution on the current CPU, causing additional overhead and fallback to slower execution paths. This highlights that quantization benefits depend heavily on hardware capabilities.

| Model | File size (MB) | Avg latency (ms) | Speedup vs PyTorch |
|---|---|---|---|
| PyTorch (FP32) | 42.81 | 73.99 | 1.00× |
| ONNX (FP32) | 42.81 | 26.00 | 2.85× |
| ONNX (INT8) | 10.75 | 354.41 | 0.21× |